# 01 Load Raw +Tour Dataset

This notebook replaces `01_load_raw_dataset.py`.

Purpose:
- Start PySpark on Windows.
- Load files from `Datasets/Raw`.
- Show schema, sample rows, row count, and columns.
- Create basic POI/user/theme summaries.

Expected location:

```text
D:\repogit\Bigdata\plustour\notebooks\01_load_raw_dataset.ipynb
```

Run this notebook using your `.venv` kernel in VS Code.

## 1. Configure Python path for PySpark on Windows

This prevents the common error where Spark calls the Microsoft Store `python` alias instead of your `.venv` Python.

In [1]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python executable:")
print(sys.executable)

Python executable:
d:\repogit\Bigdata\plusTour\.venv\Scripts\python.exe


## 2. Start Spark

In [2]:
import os
import sys
from pyspark.sql import SparkSession


def create_spark_session(app_name="PlusTourPySpark"):
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

    spark = (
        SparkSession.builder
        .appName(app_name)
        .master("local[*]")
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    return spark

## 3. Set project root and raw dataset path

Because this notebook is inside the `notebooks` folder, the project root is usually one level above the current folder.

In [4]:
import sys
import importlib
from pathlib import Path

current_path = Path.cwd()

if current_path.name.lower() == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

importlib.invalidate_caches()

from src.spark.spark_session import create_spark_session

spark = create_spark_session("PlusTourLoadRawDataset")

print("Project root:", project_root)
print("Python executable:", sys.executable)
print("Spark version:", spark.version)
print("Spark started successfully.")

Project root: d:\repogit\Bigdata\plusTour
Python executable: d:\repogit\Bigdata\plusTour\.venv\Scripts\python.exe
Spark version: 3.5.1
Spark started successfully.


## 4. Defining project and dataset path

This cell identifies the project root folder and creates the path to the raw +Tour dataset.  
Because the notebook is stored inside the `notebooks` folder, the project root is set to one level above the current working directory.  
The variable `raw_data_path` is then used by later cells to locate and load files from `Datasets/Raw`.

In [8]:
from pathlib import Path

current_path = Path.cwd()

if current_path.name.lower() == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

raw_data_path = project_root / "Datasets" / "Raw"

print("Project root:", project_root)
print("Raw data path:", raw_data_path)
print("Raw data folder exists:", raw_data_path.exists())

Project root: d:\repogit\Bigdata\plusTour
Raw data path: d:\repogit\Bigdata\plusTour\Datasets\Raw
Raw data folder exists: True


## 5. List raw dataset files

Use this to confirm Spark is reading from the correct folder.

In [9]:
if raw_data_path.exists():
    all_items = list(raw_data_path.rglob("*"))
    files = [p for p in all_items if p.is_file()]

    print("Total files found:", len(files))
    print("First 30 files:")
    for file in files[:30]:
        print(file.relative_to(project_root))
else:
    raise FileNotFoundError(f"Cannot find raw data folder: {raw_data_path}")

Total files found: 39
First 30 files:
Datasets\Raw\Athens\distanceMatrix.json
Datasets\Raw\Athens\POIs.csv
Datasets\Raw\Athens\touristsVisits.csv
Datasets\Raw\Barcelona\distanceMatrix.json
Datasets\Raw\Barcelona\POIs.csv
Datasets\Raw\Barcelona\touristsVisits.csv
Datasets\Raw\Budapest\distanceMatrix.json
Datasets\Raw\Budapest\POIs.csv
Datasets\Raw\Budapest\touristsVisits.csv
Datasets\Raw\Edinburgh\distanceMatrix.json
Datasets\Raw\Edinburgh\POIs.csv
Datasets\Raw\Edinburgh\touristsVisits.csv
Datasets\Raw\Glasgow\distanceMatrix.json
Datasets\Raw\Glasgow\POIs.csv
Datasets\Raw\Glasgow\touristsVisits.csv
Datasets\Raw\London\distanceMatrix.json
Datasets\Raw\London\POIs.csv
Datasets\Raw\London\touristsVisits.csv
Datasets\Raw\Madrid\distanceMatrix.json
Datasets\Raw\Madrid\POIs.csv
Datasets\Raw\Madrid\touristsVisits.csv
Datasets\Raw\Melbourne\distanceMatrix.json
Datasets\Raw\Melbourne\POIs.csv
Datasets\Raw\Melbourne\touristsVisits.csv
Datasets\Raw\NewDelhi\distanceMatrix.json
Datasets\Raw\NewDelh

## 6. Load raw dataset

This reads all CSV files under `Datasets/Raw`, including nested folders.

In [10]:
import pandas as pd
from pathlib import Path

csv_files = list(raw_data_path.rglob("*.csv"))

print("CSV files found:", len(csv_files))

for f in csv_files[:10]:
    print(f)

pdf_list = []

for file in csv_files:
    temp_pdf = pd.read_csv(file)
    temp_pdf["source_file"] = file.name
    pdf_list.append(temp_pdf)

pdf_raw = pd.concat(pdf_list, ignore_index=True)

df_raw = spark.createDataFrame(pdf_raw)

print("Raw dataset loaded into Spark.")
df_raw.printSchema()
df_raw.show(10, truncate=False)

CSV files found: 26
d:\repogit\Bigdata\plusTour\Datasets\Raw\Athens\POIs.csv
d:\repogit\Bigdata\plusTour\Datasets\Raw\Athens\touristsVisits.csv
d:\repogit\Bigdata\plusTour\Datasets\Raw\Barcelona\POIs.csv
d:\repogit\Bigdata\plusTour\Datasets\Raw\Barcelona\touristsVisits.csv
d:\repogit\Bigdata\plusTour\Datasets\Raw\Budapest\POIs.csv
d:\repogit\Bigdata\plusTour\Datasets\Raw\Budapest\touristsVisits.csv
d:\repogit\Bigdata\plusTour\Datasets\Raw\Edinburgh\POIs.csv
d:\repogit\Bigdata\plusTour\Datasets\Raw\Edinburgh\touristsVisits.csv
d:\repogit\Bigdata\plusTour\Datasets\Raw\Glasgow\POIs.csv
d:\repogit\Bigdata\plusTour\Datasets\Raw\Glasgow\touristsVisits.csv
Raw dataset loaded into Spark.
root
 |-- poiID: long (nullable = true)
 |-- poiName: string (nullable = true)
 |-- poiLat: double (nullable = true)
 |-- poiLon: double (nullable = true)
 |-- poiTheme: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- photoID: double (nullable = true)
 |-- userID: string (nullable = tr

## 7. Inspect schema, sample rows, row count, and columns

In [11]:
print("=== Schema ===")
df_raw.printSchema()

print("=== Sample Data ===")
df_raw.show(10, truncate=False)

print("=== Row Count ===")
print(df_raw.count())

print("=== Columns ===")
for col_name in df_raw.columns:
    print(col_name)

=== Schema ===
root
 |-- poiID: long (nullable = true)
 |-- poiName: string (nullable = true)
 |-- poiLat: double (nullable = true)
 |-- poiLon: double (nullable = true)
 |-- poiTheme: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- photoID: double (nullable = true)
 |-- userID: string (nullable = true)
 |-- dateTaken: double (nullable = true)
 |-- seqID: double (nullable = true)

=== Sample Data ===
+-----+------------------------------+----------+----------+----------+-----------+-------+------+---------+-----+
|poiID|poiName                       |poiLat    |poiLon    |poiTheme  |source_file|photoID|userID|dateTaken|seqID|
+-----+------------------------------+----------+----------+----------+-----------+-------+------+---------+-----+
|1    |Acropolis_of_Athens           |37.9715323|23.7257492|Historical|POIs.csv   |NaN    |NaN   |NaN      |NaN  |
|2    |Acropolis_Museum              |37.9684499|23.7285227|Museum    |POIs.csv   |NaN    |NaN   |NaN      |NaN

## 8. Basic POI summary

This only runs if the expected columns exist.

In [13]:
from pyspark.sql.functions import count, countDistinct

cols = df_raw.columns

if "poiID" in cols and "userID" in cols:
    poi_group_cols = ["poiID"]

    if "poiName" in cols:
        poi_group_cols.append("poiName")

    if "poiTheme" in cols:
        poi_group_cols.append("poiTheme")

    poi_summary = (
        df_raw
        .groupBy(*poi_group_cols)
        .agg(
            count("*").alias("record_count"),
            countDistinct("userID").alias("unique_users")
        )
        .orderBy("record_count", ascending=False)
    )

    poi_summary.show(20, truncate=False)
else:
    print("Cannot create POI summary because poiID and/or userID is missing.")
    print("Available columns:", cols)

+-----+-------+-------------+------------+------------+
|poiID|poiName|poiTheme     |record_count|unique_users|
+-----+-------+-------------+------------+------------+
|5    |NaN    |Entertainment|10610       |557         |
|2    |NaN    |Park         |9790        |122         |
|1    |NaN    |Historical   |6967        |854         |
|16   |NaN    |Museum       |5847        |423         |
|17   |NaN    |Historical   |5530        |493         |
|9    |NaN    |Cultural     |4730        |517         |
|11   |NaN    |Cultural     |4139        |269         |
|2    |NaN    |Museum       |4042        |257         |
|29   |NaN    |Museum       |3835        |402         |
|16   |NaN    |Precinct     |3831        |270         |
|29   |NaN    |Structure    |3826        |596         |
|1    |NaN    |Sport        |3752        |158         |
|22   |NaN    |Beach        |3603        |346         |
|21   |NaN    |Beach        |3591        |309         |
|16   |NaN    |Amusement    |3553        |377   

## 9. Basic theme summary

In [14]:
if "poiTheme" in cols:
    aggregations = [count("*").alias("records")]

    if "poiID" in cols:
        aggregations.append(countDistinct("poiID").alias("unique_pois"))

    if "userID" in cols:
        aggregations.append(countDistinct("userID").alias("unique_users"))

    theme_summary = (
        df_raw
        .groupBy("poiTheme")
        .agg(*aggregations)
        .orderBy("records", ascending=False)
    )

    theme_summary.show(30, truncate=False)
else:
    print("Cannot create theme summary because poiTheme column is missing.")

+----------------+-------+-----------+------------+
|poiTheme        |records|unique_pois|unique_users|
+----------------+-------+-----------+------------+
|Museum          |56513  |28         |3246        |
|Historical      |30599  |23         |2453        |
|Shopping        |24571  |25         |2292        |
|Cultural        |23760  |13         |1880        |
|Entertainment   |23491  |15         |1771        |
|Park            |20218  |26         |1337        |
|Structure       |16975  |20         |1603        |
|Transport       |9800   |25         |1512        |
|Sport           |8199   |8          |382         |
|Amusement       |8128   |14         |697         |
|Beach           |7400   |6          |548         |
|Palace          |5283   |7          |597         |
|Education       |5054   |10         |404         |
|Precinct        |4475   |4          |320         |
|Zoo             |3279   |6          |222         |
|Institutions    |3237   |12         |419         |
|Parks and s

## 10. Stop Spark when finished

Only run this at the end of your work.

In [15]:
spark.stop()